# Vocabulary Tools
> Vocabulary tools for working with semantic data sources.

This module provides tools for LLM agents to work with semantic vocabularies and JSON-LD data.
It augments the retriever and memory modules with agent-centric functions for working with
vocabularies, handling JSON-LD contexts, and navigating semantic data sources.

Key features:
- Support for vocabularies that don't follow standard linked data principles
- Multi-level support for different types of vocabularies
- Helper functions for creating and working with datasets
- Tools for compacting and expanding JSON-LD data

In [ ]:
#| default_exp vocabtools

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json
import httpx
from pyld import jsonld
from typing import Dict, List, Optional, Any, Union
from rdflib import Graph
import copy
import uuid

In [ ]:
#| export
# Enhanced registry of vocabulary information with alternative URIs
VOCABULARY_REGISTRY = {
    "schema": {
        "uri": "https://schema.org/",
        "alternative_uris": ["http://schema.org/", "https://schema.org", "http://schema.org"],
        "prefix": "schema",
        "title": "Schema.org",
        "description": "Schema.org vocabulary for structured data on the internet",
        "version": "latest",
        "publisher": "Schema.org Community",
        "support_level": "direct",
        "resources": {
            "ttl": "https://schema.org/version/latest/schemaorg-current-https.ttl",
            "context": "https://schema.org/docs/jsonldcontext.jsonld",
            "homepage": "https://schema.org/"
        },
        "access_patterns": {
            "primary": "link_header_to_context",
            "fallbacks": ["direct_download"]
        },
        "url_transformations": [],
        "features": {
            "inline_context": False,
            "uses_protection": False,
            "supports_scoped_contexts": True
        },
        "common_terms": ["name", "description", "url", "image"],
        "common_types": ["Thing", "Person", "Organization", "Place", "Event", "Product"],
        "related_vocabs": ["dc", "foaf"]
    },
    
    "croissant": {
        "uri": "http://mlcommons.org/croissant/",
        "alternative_uris": ["https://mlcommons.org/croissant/"],
        "prefix": "croissant",
        "title": "CROISSANT",
        "description": "Community Resource for Open and Inclusive Scholarly Sources and Artifacts for Novel Technologies",
        "version": "1.0",
        "publisher": "MLCommons",
        "support_level": "direct",
        "resources": {
            "ttl": "https://raw.githubusercontent.com/mlcommons/croissant/main/docs/croissant.ttl",
            "context": "https://raw.githubusercontent.com/mlcommons/croissant/main/docs/context.jsonld",
            "homepage": "https://github.com/mlcommons/croissant"
        },
        "access_patterns": {
            "primary": "github_raw",
            "fallbacks": []
        },
        "url_transformations": [],
        "features": {
            "inline_context": True,
            "uses_protection": False,
            "supports_scoped_contexts": False
        },
        "common_terms": ["name", "description", "creator", "license"],
        "common_types": ["Dataset", "DatasetField", "Distribution"],
        "related_vocabs": ["schema", "dcat"]
    },
    
    "ro-crate": {
        "uri": "https://w3id.org/ro/crate/1.2-DRAFT/context",
        "alternative_uris": ["https://w3id.org/ro/crate/1.1/context", "https://w3id.org/ro/crate/context"],
        "prefix": "ro",
        "title": "RO-Crate",
        "description": "Research Object Crate (RO-Crate) metadata specification",
        "version": "1.2-DRAFT",
        "publisher": "ResearchObject.org",
        "support_level": "cache",
        "resources": {
            "ttl": None,
            "context": "https://w3id.org/ro/crate/1.2-DRAFT/context",
            "homepage": "https://www.researchobject.org/ro-crate/"
        },
        "access_patterns": {
            "primary": "persistent_uri",
            "fallbacks": []
        },
        "url_transformations": [],
        "features": {
            "inline_context": False,
            "uses_protection": False,
            "supports_scoped_contexts": False
        },
        "common_terms": ["name", "description", "author", "hasPart"],
        "common_types": ["Dataset", "CreativeWork", "Person"],
        "related_vocabs": ["schema", "dc"]
    },
    
    "vc": {
        "uri": "https://www.w3.org/ns/credentials/v2",
        "alternative_uris": ["https://www.w3.org/2018/credentials/v1", "https://www.w3.org/2018/credentials"],
        "prefix": "vc",
        "title": "Verifiable Credentials Data Model",
        "description": "W3C Verifiable Credentials data model and representations",
        "version": "v2",
        "publisher": "W3C",
        "support_level": "direct",
        "resources": {
            "ttl": None,
            "context": "https://www.w3.org/ns/credentials/v2",
            "backup": "https://raw.githubusercontent.com/w3c/vc-data-model/refs/heads/main/contexts/credentials/v2",
            "homepage": "https://www.w3.org/TR/vc-data-model-2.0/"
        },
        "access_patterns": {
            "primary": "github_raw",
            "fallbacks": ["direct_download"]
        },
        "url_transformations": [],
        "features": {
            "inline_context": False,
            "uses_protection": True,
            "supports_scoped_contexts": True
        },
        "common_terms": ["issuer", "credentialSubject", "validFrom", "validUntil"],
        "common_types": ["VerifiableCredential", "VerifiablePresentation"],
        "related_vocabs": ["schema"]
    },
    
    "epcis": {
        "uri": "https://ref.gs1.org/epcis/",
        "alternative_uris": ["https://gs1.org/epcis/", "http://gs1.org/epcis/", "https://gs1.github.io/EPCIS/"],
        "prefix": "epcis",
        "title": "EPCIS",
        "description": "GS1 EPCIS standard for supply chain visibility",
        "version": "2.0",
        "publisher": "GS1",
        "support_level": "direct",
        "resources": {
            "ttl": None,
            "context": "https://raw.githubusercontent.com/gs1/EPCIS/master/JSON-LD-Context/epcis-context-root.jsonld",
            "homepage": "https://ref.gs1.org/standards/epcis/"
        },
        "access_patterns": {
            "primary": "github_raw",
            "fallbacks": []
        },
        "url_transformations": [
            {
                "pattern": r"https://gs1\.github\.io/EPCIS/(.+\.jsonld)",
                "replacement": r"https://raw.githubusercontent.com/gs1/EPCIS/master/JSON-LD-Context/\1"
            }
        ],
        "features": {
            "inline_context": False,
            "uses_protection": True,
            "supports_scoped_contexts": False
        },
        "common_terms": ["eventTime", "epcList", "bizLocation", "bizStep"],
        "common_types": ["ObjectEvent", "AggregationEvent", "TransactionEvent"],
        "related_vocabs": []
    },
    
    "dc": {
        "uri": "http://purl.org/dc/terms/",
        "alternative_uris": ["http://purl.org/dc/elements/1.1/", "http://dublincore.org/"],
        "prefix": "dc",
        "title": "Dublin Core",
        "description": "Dublin Core Metadata Initiative Terms",
        "version": "terms",
        "publisher": "DCMI",
        "support_level": "cache",
        "resources": {
            "ttl": "https://www.dublincore.org/specifications/dublin-core/dcmi-terms/dublin_core_terms.ttl",
            "context": "https://www.dublincore.org/contexts/dc-terms/",
            "homepage": "https://www.dublincore.org/"
        },
        "access_patterns": {
            "primary": "direct_download",
            "fallbacks": []
        },
        "url_transformations": [],
        "features": {
            "inline_context": False,
            "uses_protection": False,
            "supports_scoped_contexts": False
        },
        "common_terms": ["title", "creator", "date", "subject"],
        "common_types": ["Agent", "BibliographicResource", "Location"],
        "related_vocabs": ["schema", "foaf"]
    },
    
    "foaf": {
        "uri": "http://xmlns.com/foaf/0.1/",
        "alternative_uris": ["http://xmlns.com/foaf/spec/", "http://xmlns.com/foaf/"],
        "prefix": "foaf",
        "title": "Friend of a Friend",
        "description": "Friend of a Friend vocabulary for describing people and their connections",
        "version": "0.1",
        "publisher": "FOAF Project",
        "support_level": "cache",
        "resources": {
            "ttl": "http://xmlns.com/foaf/spec/index.rdf",
            "context": None,
            "homepage": "http://xmlns.com/foaf/spec/"
        },
        "access_patterns": {
            "primary": "direct_download",
            "fallbacks": []
        },
        "url_transformations": [],
        "features": {
            "inline_context": False,
            "uses_protection": False,
            "supports_scoped_contexts": False
        },
        "common_terms": ["name", "mbox", "knows", "homepage"],
        "common_types": ["Person", "Organization", "Document"],
        "related_vocabs": ["schema", "dc"]
    },
    
    "dcat": {
        "uri": "http://www.w3.org/ns/dcat#",
        "alternative_uris": ["https://www.w3.org/ns/dcat#", "http://www.w3.org/ns/dcat"],
        "prefix": "dcat",
        "title": "Data Catalog Vocabulary",
        "description": "W3C Data Catalog Vocabulary for describing datasets",
        "version": "2",
        "publisher": "W3C",
        "support_level": "cache",
        "resources": {
            "ttl": "https://www.w3.org/ns/dcat2.ttl",
            "context": None,
            "homepage": "https://www.w3.org/TR/vocab-dcat-2/"
        },
        "access_patterns": {
            "primary": "direct_download",
            "fallbacks": []
        },
        "url_transformations": [],
        "features": {
            "inline_context": False,
            "uses_protection": False,
            "supports_scoped_contexts": False
        },
        "common_terms": ["distribution", "downloadURL", "accessURL", "theme"],
        "common_types": ["Dataset", "Distribution", "Catalog"],
        "related_vocabs": ["dc", "schema"]
    }
}


In [ ]:
#| export
# Enhanced collision strategies for vocabulary combinations
COLLISION_STRATEGIES = {
    # VC + EPCIS: Use property scoping to place EPCIS data within credentialSubject
    ("vc", "epcis"): {
        "strategy": "property_scoped",
        "primary": "vc",
        "property": "credentialSubject",
        "secondary": "epcis",
        "description": "Places EPCIS data within the VC credentialSubject property"
    },
    
    # Schema.org + Dublin Core: Use graph partitioning with schema as primary
    ("schema", "dc"): {
        "strategy": "graph_partition",
        "primary": "schema",
        "description": "Partitions the graph with Schema.org as the primary vocabulary"
    },
    
    # Schema.org + FOAF: Use property-specific mapping
    ("schema", "foaf"): {
        "strategy": "property_mapping",
        "mappings": {
            "foaf:name": "schema:name",
            "foaf:Person": "schema:Person",
            "foaf:knows": "schema:knows"
        },
        "description": "Maps FOAF properties to Schema.org equivalents"
    },
    
    # RO-Crate + DCAT: Use nested contexts
    ("ro-crate", "dcat"): {
        "strategy": "nested_contexts",
        "outer": "ro-crate",
        "inner": "dcat",
        "description": "Nests DCAT context within RO-Crate"
    },
    
    # PROV + Schema: Use context versioning
    ("prov", "schema"): {
        "strategy": "context_versioning",
        "context_version": "1.1",
        "description": "Uses JSON-LD 1.1 features to handle these vocabularies"
    },
    
    # SHACL + OWL: Keep separate with explicit references
    ("shacl", "owl"): {
        "strategy": "separate_graphs",
        "description": "Maintains separate graphs with explicit cross-references"
    },
    
    # Default strategy for any vocab with @protected terms
    ("*", "*_protected"): {
        "strategy": "graph_partition",
        "description": "Default strategy when any vocabulary uses @protected terms"
    }
}


In [ ]:
#| export
class VocabularyManager:
    "Manages vocabulary contexts and document loading"
    def __init__(self, registry=None): 
        self.registry = registry or VOCABULARY_REGISTRY
        self.context_cache = {}
        self.loaded_vocabs = set()
    
    def handle_direct_support(self, url, vocab_info):
        "Handle vocabularies that need direct intervention"
        print(f"--- [Direct] handle_direct_support for: {url}") # DEBUG
        if url in self.context_cache:
            print(f"--- [Direct] Cache HIT for {url}") # DEBUG
            return self.context_cache[url]
        print(f"--- [Direct] Cache MISS for {url}") # DEBUG

        context_location = vocab_info.get("resources", {}).get("context")
        if not context_location:
            print(f"--- [Direct] No context location found for {url}. Creating minimal.") # DEBUG
            return self._create_minimal_context(url, vocab_info)
        print(f"--- [Direct] Context location: {context_location}") # DEBUG

        try:
            print(f"--- [Direct] Fetching {context_location}...") # DEBUG
            # Use httpx with redirects and timeout
            response = httpx.get(context_location, follow_redirects=True, timeout=10.0)
            print(f"--- [Direct] Status code for {context_location}: {response.status_code}") # DEBUG
            response.raise_for_status() # Raise HTTPError for bad responses (4xx or 5xx)

            try:
                context_data = response.json()
                print(f"--- [Direct] Successfully parsed JSON from {context_location}") # DEBUG
                # The documentUrl should be the originally requested or primary vocab URI
                result = {'contextUrl': None, 'documentUrl': url, 'document': context_data}
                self.context_cache[url] = result # Cache under the URL passed to this function (e.g., the primary URI)
                return result
            except json.JSONDecodeError as e:
                print(f"--- [Direct] Error parsing context JSON from {context_location}: {e}") # DEBUG
                backup_location = vocab_info.get("resources", {}).get("backup")
                if backup_location:
                    print(f"--- [Direct] Trying backup location: {backup_location}") # DEBUG
                    # Pass the original/primary URL 'url' to the backup function too
                    return self._try_backup_location(url, backup_location)
                else:
                    print(f"--- [Direct] No backup location available.") # DEBUG
                    # Return None to indicate failure, allowing vocab_aware_loader to continue trying things
                    return None

        # Catch specific httpx errors
        except httpx.RequestError as e:
            print(f"--- [Direct] HTTP Request Error fetching context from {context_location}: {e}") # DEBUG
        except httpx.HTTPStatusError as e:
             print(f"--- [Direct] HTTP Status Error fetching context from {context_location}: {e.response.status_code} - {e.response.text[:200]}...") # DEBUG
        except Exception as e:
            # Catch any other unexpected errors during fetch/processing
            print(f"--- [Direct] Unexpected Error fetching context from {context_location}: {type(e).__name__} - {e}") # DEBUG

        # If fetch failed, try fallbacks
        fallbacks = vocab_info.get("access_patterns", {}).get("fallbacks", [])
        print(f"--- [Direct] Trying fallbacks: {fallbacks}") # DEBUG
        for pattern in fallbacks:
            # Pass the original/primary URL 'url' to the pattern function
            result = self._try_access_pattern(url, vocab_info, pattern)
            if result:
                print(f"--- [Direct] Fallback pattern '{pattern}' succeeded for {url}.") # DEBUG
                return result

        # Fallback to minimal context if all else failed
        print(f"--- [Direct] All fetch attempts failed for {url}. Creating minimal.") # DEBUG
        return self._create_minimal_context(url, vocab_info)
    
    def _try_backup_location(self, url, backup_location):
        "Try fetching from backup location"
        try:
            backup_resp = httpx.get(backup_location, follow_redirects=True, timeout=10.0)
            backup_resp.raise_for_status()
            backup_data = backup_resp.json()
            result = {'contextUrl': None, 'documentUrl': url, 'document': backup_data}
            self.context_cache[url] = result # Cache under original/primary URL
            print(f"--- [Backup] Success fetching from {backup_location} for {url}") # DEBUG
            return result
        except httpx.RequestError as backup_e:
            print(f"--- [Backup] HTTP Request Error fetching from {backup_location}: {backup_e}") # DEBUG
        except httpx.HTTPStatusError as backup_e:
            print(f"--- [Backup] HTTP Status Error fetching from {backup_location}: {backup_e.response.status_code}") # DEBUG
        except json.JSONDecodeError as backup_e:
            print(f"--- [Backup] JSON Decode Error fetching from {backup_location}: {backup_e}") # DEBUG
        except Exception as backup_e:
            print(f"--- [Backup] Unexpected Error fetching from {backup_location}: {type(backup_e).__name__} - {backup_e}") # DEBUG
        return None
    
    def _try_access_pattern(self, url, vocab_info, pattern):
        "Try a specific access pattern for context retrieval"
        print(f"--- [Pattern] Trying pattern '{pattern}' for {url}") # DEBUG
        if pattern == "direct_download":
            # This pattern usually means trying the 'url' itself directly
            try:
                response = httpx.get(url, follow_redirects=True, timeout=10.0, headers={'Accept': 'application/ld+json, application/json'})
                response.raise_for_status()
                try:
                    context_data = response.json()
                    result = {'contextUrl': None, 'documentUrl': url, 'document': context_data}
                    self.context_cache[url] = result
                    print(f"--- [Pattern] Direct download succeeded for {url}") # DEBUG
                    return result
                except json.JSONDecodeError as e:
                    print(f"--- [Pattern] Direct download JSON error for {url}: {e}") # DEBUG
                    pass # Fall through
            except httpx.RequestError as e:
                print(f"--- [Pattern] Direct download HTTP Request error for {url}: {e}") # DEBUG
            except httpx.HTTPStatusError as e:
                print(f"--- [Pattern] Direct download HTTP Status error for {url}: {e.response.status_code}") # DEBUG
            except Exception as e:
                 print(f"--- [Pattern] Direct download unexpected error for {url}: {type(e).__name__} - {e}") # DEBUG
        # Add other pattern handling here if needed
        return None
    
    def _create_minimal_context(self, url, vocab_info):
        "Create a minimal context when all else fails"
        print(f"--- [Minimal] Creating minimal context for {url}") # DEBUG
        # Use inline context if available (though less likely for minimal fallback)
        if vocab_info.get("features", {}).get("inline_context", False):
            default_context = {"@vocab": vocab_info.get("uri", url)}
            common_terms = vocab_info.get("common_terms", [])
            for term in common_terms: default_context[term] = f"{vocab_info.get('uri', url)}{term}"
        else:
            default_context = {"@vocab": vocab_info.get("uri", url)} # Simplest fallback
        
        result = {'contextUrl': None, 'documentUrl': url, 'document': default_context}
        self.context_cache[url] = result # Cache the minimal context
        return result
    
    def handle_cache_support(self, url, vocab_info, default_loader):
        "Handle vocabularies that can be dereferenced but benefit from caching"
        print(f"--- [Cache] handle_cache_support for: {url}") # DEBUG
        if url in self.context_cache:
            print(f"--- [Cache] Cache HIT for {url}") # DEBUG
            return self.context_cache[url]
        print(f"--- [Cache] Cache MISS for {url}") # DEBUG
        
        transformed_url = self.apply_url_transformations(url, vocab_info)
        
        try:
            print(f"--- [Cache] Trying default_loader for {transformed_url} (original: {url})") # DEBUG
            result = default_loader(transformed_url, {})
            print(f"--- [Cache] default_loader SUCCEEDED for {transformed_url}") # DEBUG
            # Ensure documentUrl reflects the originally requested URL if transformed
            if transformed_url != url: result['documentUrl'] = url
            self.context_cache[url] = result # Cache under original URL
            if transformed_url != url: self.context_cache[transformed_url] = result # Cache under transformed URL too
            return result
        except Exception as e:
            print(f"--- [Cache] default_loader FAILED for {transformed_url}: {e}") # DEBUG
            
            context_location = vocab_info.get("resources", {}).get("context")
            if context_location and context_location != url and context_location != transformed_url:
                print(f"--- [Cache] Trying default_loader for explicit context_location: {context_location}") # DEBUG
                try:
                    result = default_loader(context_location, {})
                    print(f"--- [Cache] default_loader SUCCEEDED for context_location {context_location}") # DEBUG
                    result['documentUrl'] = url # Set documentUrl to the original requested URL
                    self.context_cache[url] = result # Cache under original URL
                    self.context_cache[context_location] = result # Cache under context location URL too
                    return result
                except Exception as inner_e:
                    print(f"--- [Cache] default_loader FAILED for context_location {context_location}: {inner_e}") # DEBUG
        
        print(f"--- [Cache] All attempts failed for {url}. Creating minimal.") # DEBUG
        return self._create_minimal_context(url, vocab_info)
    
    def apply_url_transformations(self, url, vocab_info):
        "Apply URL transformations defined in vocabulary info"
        import re
        
        transformations = vocab_info.get("url_transformations", [])
        current_url = url
        for transform in transformations:
            pattern = transform.get("pattern")
            replacement = transform.get("replacement")
            if pattern and replacement:
                try:
                    # Use re.fullmatch to only transform if the whole URL matches the pattern
                    match = re.fullmatch(pattern, current_url)
                    if match:
                        transformed_url = re.sub(pattern, replacement, current_url)
                        if transformed_url != current_url:
                            print(f"--- [Transform] Applied transformation: {current_url} -> {transformed_url}") # DEBUG
                            current_url = transformed_url
                            # If a transformation is applied, maybe stop? Or allow chaining? Assuming chain for now.
                except Exception as e:
                    print(f"--- [Transform] Error applying URL transformation pattern '{pattern}': {e}") # DEBUG
        
        return current_url # Return the potentially transformed URL
    
    def handle_discovery_support(self, url, default_loader):
        "Handle unknown vocabularies by attempting to discover their structure"
        print(f"--- [Discover] handle_discovery_support for: {url}") # DEBUG
        if url in self.context_cache:
            print(f"--- [Discover] Cache HIT for {url}") # DEBUG
            return self.context_cache[url]
        print(f"--- [Discover] Cache MISS for {url}") # DEBUG
        
        try:
            print(f"--- [Discover] Trying default_loader for {url}") # DEBUG
            result = default_loader(url, {})
            print(f"--- [Discover] default_loader SUCCEEDED for {url}") # DEBUG
            self.context_cache[url] = result
            return result
        except Exception as e:
            print(f"--- [Discover] default_loader FAILED for {url}: {e}") # DEBUG
        
        # Common URL variations to try
        variations = [
            f"{url}/context",
            f"{url.rstrip('/')}/context.jsonld",
            f"{url}/latest/context",
            f"{url.rstrip('/')}/.well-known/context.jsonld"
        ]
        
        print(f"--- [Discover] Trying variations for {url}: {variations}") # DEBUG
        for variation in variations:
            try:
                print(f"--- [Discover] Trying variation: {variation}") # DEBUG
                result = default_loader(variation, {})
                print(f"--- [Discover] Variation {variation} SUCCEEDED") # DEBUG
                result['documentUrl'] = url # Set documentUrl to original
                self.context_cache[url] = result # Cache under original URL
                self.context_cache[variation] = result # Cache under variation URL too
                return result
            except Exception as e:
                print(f"--- [Discover] Variation {variation} FAILED: {e}") # DEBUG
        
        print(f"--- [Discover] All attempts failed for {url}. Creating minimal.") # DEBUG
        return self._create_minimal_context(url, {"uri": url}) # Pass minimal info for minimal context creation
    
    def create_document_loader(self):
        "Create a document loader that handles vocabularies at different support levels"
        from pyld import jsonld
        default_loader = jsonld.get_document_loader()

        def vocab_aware_loader(url, options=None):
            "Custom document loader that handles different vocabulary support levels"
            print(f"\n--- vocab_aware_loader called for: {url}") # DEBUG
            options = options or {}

            # First check for direct cache hit on the exact URL requested
            if url in self.context_cache:
                print(f"--- Cache HIT for {url}") # DEBUG
                return self.context_cache[url]
            print(f"--- Cache MISS for {url}") # DEBUG

            # Check registry for alternative/primary URIs
            matched_vocab_info = None
            matched_vocab_name = None
            use_primary_uri = None

            for vocab_name, vocab_info in self.registry.items():
                primary_uri = vocab_info.get("uri")
                if not primary_uri: continue

                # Check alternative URIs first
                alt_uris = vocab_info.get("alternative_uris", [])
                if url in alt_uris:
                    print(f"--- Matched ALT URI for {vocab_name}: {url} -> using PRIMARY {primary_uri}") # DEBUG
                    matched_vocab_info = vocab_info
                    matched_vocab_name = vocab_name
                    use_primary_uri = primary_uri
                    break # Found a match, stop checking registry

                # Check for exact match with primary URI (if not an alt URI)
                if url == primary_uri:
                    print(f"--- Matched PRIMARY URI for {vocab_name}: {url}") # DEBUG
                    matched_vocab_info = vocab_info
                    matched_vocab_name = vocab_name
                    use_primary_uri = url # Use the matched URL itself
                    break # Found a match

            # If a vocabulary was matched (either primary or alternative)
            if matched_vocab_info:
                self.loaded_vocabs.add(matched_vocab_name)
                target_uri = use_primary_uri # This is the URI we'll use for fetching/cache check

                # Check cache using the target URI (could be primary or the original URL if it was primary)
                if target_uri in self.context_cache:
                    print(f"--- Cache HIT for TARGET {target_uri} (original: {url})") # DEBUG
                    # Return the cached context, but ensure it's also cached under the original URL
                    cached_result = self.context_cache[target_uri]
                    if url not in self.context_cache: self.context_cache[url] = cached_result
                    return cached_result
                print(f"--- Cache MISS for TARGET {target_uri} (original: {url})") # DEBUG

                # Handle based on support level using the target URI
                support_level = matched_vocab_info.get("support_level", "discover")
                print(f"--- Handling {matched_vocab_name} (level: {support_level}) using TARGET {target_uri}") # DEBUG
                handler_result = None
                if support_level == "direct": handler_result = self.handle_direct_support(target_uri, matched_vocab_info)
                elif support_level == "cache": handler_result = self.handle_cache_support(target_uri, matched_vocab_info, default_loader)
                else: handler_result = self.handle_discovery_support(target_uri, default_loader)

                if handler_result:
                    print(f"--- Handler SUCCEEDED for TARGET {target_uri} (original: {url})") # DEBUG
                    # IMPORTANT: Cache the result under the ORIGINAL requested URL as well.
                    # The handler should have cached under target_uri already.
                    self.context_cache[url] = handler_result
                    # Ensure the documentUrl in the result reflects the original request 'url'
                    handler_result['documentUrl'] = url
                    return handler_result
                else:
                    # If the handler failed for the target URI, log it and fall through to other methods (variation, transform, default)
                    print(f"--- Handler FAILED for TARGET {target_uri} (original: {url}). Falling through.") # DEBUG
            
            # Handle URL variations (e.g. trailing slash) - Check AFTER specific registry matches fail or fall through
            if url.startswith(('http://', 'https://')):
                alt_url = url + '/' if not url.endswith('/') else url[:-1]
                print(f"--- Checking variation: {alt_url}") # DEBUG
                if alt_url in self.context_cache:
                    print(f"--- Cache HIT for variation {alt_url}") # DEBUG
                    # Cache under the original URL too for next time
                    self.context_cache[url] = self.context_cache[alt_url]
                    return self.context_cache[alt_url]
                # Optional: Could re-run the registry check logic for alt_url here if needed as a deeper fallback

            # Check for URL transformations across all vocabularies (if no direct match handled it)
            for vocab_name, vocab_info in self.registry.items():
                transformed_url = self.apply_url_transformations(url, vocab_info)
                if transformed_url != url:
                    # This transformation logic seems okay, uses default_loader
                    print(f"--- URL Transformation applied by {vocab_name}: {url} -> {transformed_url}") # DEBUG
                    try:
                        print(f"--- Trying default_loader with transformed URL: {transformed_url}") # DEBUG
                        result = default_loader(transformed_url, options)
                        # Ensure documentUrl reflects the original request
                        result['documentUrl'] = url
                        self.context_cache[url] = result # Cache under original URL
                        if transformed_url not in self.context_cache: self.context_cache[transformed_url] = result # Cache under transformed too
                        print(f"--- default_loader with transformed URL SUCCEEDED") # DEBUG
                        return result
                    except Exception as e:
                        print(f"--- default_loader with transformed URL FAILED: {e}") # DEBUG
                        # Continue checking other transformations or fall through
                        pass

            # No match found in registry, variations, or transformations - use default loader
            print(f"--- No custom handling for {url}. Falling back to default_loader.") # DEBUG
            try:
                result = default_loader(url, options)
                # Optionally cache results from default loader too
                # self.context_cache[url] = result
                print(f"--- default_loader SUCCEEDED for {url}") # DEBUG
                return result
            except Exception as e:
                 print(f"--- default_loader FAILED for {url}: {type(e).__name__} - {e}") # DEBUG
                 # Re-raise the original error from pyld's default loader
                 raise jsonld.JsonLdError(
                    'Could not retrieve a JSON-LD document from the URL.',
                    'jsonld.LoadDocumentError', {'code': 'loading document failed', 'url': url},
                    cause=e)

        return vocab_aware_loader
    
    
    def get_loaded_vocabularies(self):
        "Get list of vocabularies that have been loaded"
        return list(self.loaded_vocabs)
    
    def get_vocabulary_info(self, vocab_name):
        "Get detailed information about a vocabulary"
        return self.registry.get(vocab_name) # Use .get for safer access
    
    # Note: detect_vocabularies_in_context was defined outside the class in the notebook
    # If it should be part of the class, it needs 'self' and indentation.
    # Keeping it outside based on the provided notebook structure.

#| export
# Static method version if preferred outside the class instance
def detect_vocabularies_in_context(context_list, registry=None):
    "Detect which vocabularies are used in a given context list"
    reg = registry or VOCABULARY_REGISTRY # Use provided or global registry
    vocabs = set() # Use a set to avoid duplicates initially
    
    if not isinstance(context_list, list): context_list = [context_list]
    
    for ctx in context_list:
        if isinstance(ctx, str):
            for name, info in reg.items():
                uris = [info.get("uri")] + info.get("alternative_uris", [])
                if any(ctx == u for u in uris if u): # Check exact match against primary and alternatives
                    vocabs.add(name)
                    break # Found match for this context string
                # Optional: Add startswith check if needed, but exact match is safer for context URLs
                # elif any(ctx.startswith(u) for u in uris if u):
                #     vocabs.add(name)
                #     break
        elif isinstance(ctx, dict):
            # Check @vocab
            vocab_uri = ctx.get("@vocab")
            if vocab_uri:
                for name, info in reg.items():
                    uris = [info.get("uri")] + info.get("alternative_uris", [])
                    if any(vocab_uri == u for u in uris if u):
                        vocabs.add(name)
                        break # Found match for @vocab
            
            # Check prefixes defined in the context dict against registry prefixes
            for name, info in reg.items():
                prefix = info.get("prefix")
                if prefix and prefix in ctx: # Check if the prefix key exists in the context dict
                    # More robust check: ensure the value associated with the prefix resembles a URI/namespace
                    prefix_val = ctx.get(prefix)
                    if isinstance(prefix_val, str) and ('http' in prefix_val or ':' in prefix_val):
                         # Could also check if prefix_val matches info['uri'] or alt_uris
                         vocabs.add(name)

    return list(vocabs)

In [ ]:
#| export
# Create a singleton instance
_manager = VocabularyManager()

In [ ]:
#| export
def test_fixed_document_loader():
    "Test our fixed document loader with schema.org"
    from pyld import jsonld
    from cogitarelink.vocabtools import _manager, VOCABULARY_REGISTRY
    import types
    
    # Register our fixed loader
    print("Registering fixed vocabulary-aware loader...")
    new_loader = _manager.create_document_loader() # This correctly calls the method on the instance
    jsonld.set_document_loader(new_loader)
    
    # Test with schema.org URLs
    urls_to_test = [
        "http://schema.org/",
        "https://schema.org/",
        "http://schema.org",
        "https://schema.org"
    ]
    
    print("\nTesting fixed document loader with schema.org URLs:")
    for url in urls_to_test:
        print(f"\nTesting URL: {url}")
        try:
            result = new_loader(url)
            print(f"Success! Document URL: {result.get('documentUrl')}")
            print(f"Context URL: {result.get('contextUrl')}")
            doc_size = len(str(result.get('document', '')))
            print(f"Document size: {doc_size} chars")
        except Exception as e:
            print(f"Error: {e}")
    
    # Test expanding a simple schema.org entity
    print("\nTesting expansion of schema.org entity with HTTP URL:")
    test_entity = {
        "@context": "http://schema.org/",
        "@type": "Person",
        "name": "Test Person"
    }
    
    try:
        expanded = jsonld.expand(test_entity)
        print(f"Successfully expanded: {expanded}")
    except Exception as e:
        print(f"Expansion error: {e}")
    
    # Test with memory add method
    print("\nTesting with SemanticMemory.add():")
    # Ensure SemanticMemory is importable or defined before this test runs
    try:
        from cogitarelink.memory import SemanticMemory
        mem = SemanticMemory()
        
        test_data = {
            "@context": "http://schema.org/",
            "@type": "Person",
            "name": "John Doe",
            "jobTitle": "Software Developer"
        }
        
        try:
            result = mem.add(test_data) # Assuming add method exists and works with dict
            print(f"Successfully added entity with result type: {type(result)}")
            if isinstance(result, list) and len(result) > 0:
                print(f"First result ID: {result[0].get('@id', 'No ID')}")
            elif isinstance(result, dict):
                print(f"Result ID: {result.get('@id', 'No ID')}")
        except Exception as e:
            print(f"Add error: {e}")
            
    except ImportError:
        print("Skipping SemanticMemory test: cogitarelink.memory not found or SemanticMemory not defined.")
        
    
    return "Test complete"

In [ ]:
test_fixed_document_loader()

Registering fixed vocabulary-aware loader...

Testing fixed document loader with schema.org URLs:

Testing URL: http://schema.org/
Error: ('Could not retrieve a JSON-LD document from the URL.',)
Type: jsonld.LoadDocumentError
Code: loading document failed
Cause: Expecting value: line 2 column 1 (char 1)  File "/Users/cvardema/dev/git/LA3D/cogitarelink/.venv/lib/python3.12/site-packages/pyld/documentloader/requests.py", line 72, in loader
    'document': response.json()
                ^^^^^^^^^^^^^^^
  File "/Users/cvardema/dev/git/LA3D/cogitarelink/.venv/lib/python3.12/site-packages/requests/models.py", line 978, in json
    raise RequestsJSONDecodeError(e.msg, e.doc, e.pos)


Testing URL: https://schema.org/
Success! Document URL: https://schema.org/
Context URL: None
Document size: 156526 chars

Testing URL: http://schema.org
Error: ('Could not retrieve a JSON-LD document from the URL.',)
Type: jsonld.LoadDocumentError
Code: loading document failed
Cause: Expecting value: line 2 col

'Test complete'

## Export functions that use the singleton manager

In [ ]:
#| export
# Export functions that use the singleton manager
def create_vocab_aware_document_loader(registry=None):
    "Create a document loader that handles vocabularies at different support levels"
    if registry: _manager.registry = registry
    return _manager.create_document_loader()

In [ ]:
#| export
def register_vocab_aware_loader():
    "Register our vocabulary-aware document loader with PyLD"
    from pyld import jsonld
    loader = create_vocab_aware_document_loader()
    jsonld.set_document_loader(loader)
    return loader

In [ ]:
#| export
def get_loaded_vocabularies():
    "Get list of vocabularies that have been loaded"
    return _manager.get_loaded_vocabularies()

In [ ]:
#| export
def get_vocabulary_info(vocab_name):
    "Get detailed information about a vocabulary"
    return _manager.get_vocabulary_info(vocab_name)


In [ ]:
#| export
def detect_vocabularies_in_context(context_list):
    """Detect which vocabularies are used in a given context list"""
    vocabs = []
    
    # Normalize input to list
    if not isinstance(context_list, list): context_list = [context_list]
    
    # Check each context against registry entries
    for ctx in context_list:
        if isinstance(ctx, str):
            # Check against all vocabulary URIs
            for vocab_name, vocab_info in VOCABULARY_REGISTRY.items():
                # Check primary URI
                if ctx == vocab_info.get("uri", "") or ctx.startswith(vocab_info.get("uri", "")):
                    vocabs.append(vocab_name)
                    continue
                    
                # Check alternative URIs if available
                for alt_uri in vocab_info.get("alternative_uris", []):
                    if ctx == alt_uri or ctx.startswith(alt_uri):
                        vocabs.append(vocab_name)
                        break
        elif isinstance(ctx, dict):
            # Check @vocab against URIs
            if "@vocab" in ctx:
                vocab_uri = ctx["@vocab"]
                for vocab_name, vocab_info in VOCABULARY_REGISTRY.items():
                    if vocab_uri == vocab_info.get("uri", "") or any(vocab_uri == alt for alt in vocab_info.get("alternative_uris", [])):
                        vocabs.append(vocab_name)
            
            # Check for prefixes in context
            for vocab_name, vocab_info in VOCABULARY_REGISTRY.items():
                prefix = vocab_info.get("prefix", "")
                if prefix and prefix in ctx:
                    vocabs.append(vocab_name)
    
    # Remove duplicates
    return list(set(vocabs))


In [ ]:
#| export
def apply_url_transformation(url):
    "Apply URL transformations from all vocabularies"
    for vocab_info in VOCABULARY_REGISTRY.values():
        transformed = _manager.apply_url_transformations(url, vocab_info)
        if transformed != url:
            return transformed
    return url


In [ ]:
#| cxport
def get_collision_strategy(context_list):
    """Get the appropriate collision strategy for these contexts"""
    vocabs = detect_vocabularies_in_context(context_list)
    
    if len(vocabs) < 2: return None  # No collision possible with just one vocabulary
    
    # Sort vocabulary names to ensure consistent lookup
    vocabs.sort()
    vocab_pair = tuple(vocabs[:2])  # Take the first two vocabs
    
    # Check for specific strategies for this vocabulary pair
    if vocab_pair in COLLISION_STRATEGIES:
        return COLLISION_STRATEGIES[vocab_pair]
    
    # Try the reversed pair
    reversed_pair = (vocab_pair[1], vocab_pair[0])
    if reversed_pair in COLLISION_STRATEGIES:
        return COLLISION_STRATEGIES[reversed_pair]
    
    # Check for wildcard strategies
    for strategy_key, strategy in COLLISION_STRATEGIES.items():
        if "*" in strategy_key:
            # Handle wildcard patterns like ("*", "*_protected")
            if strategy_key[0] == "*" and any(v.endswith(strategy_key[1][1:]) for v in vocabs):
                return strategy
            elif strategy_key[1] == "*" and any(v.startswith(strategy_key[0]) for v in vocabs):
                return strategy
    
    # Default strategy based on whether any vocabulary uses protection
    uses_protection = any(
        VOCABULARY_REGISTRY.get(v, {}).get("features", {}).get("uses_protection", False) 
        for v in vocabs
    )
    
    if uses_protection:
        return {"strategy": "graph_partition", "description": "Default for protected vocabularies"}
    else:
        return {"strategy": "scoped_context", "description": "Default for unprotected vocabularies"}


In [ ]:
#| export
def apply_collision_strategy(data, strategy):
    """Apply a collision strategy to the data"""
    if not strategy or not isinstance(data, dict) or "@context" not in data:
        return data
    
    strategy_type = strategy.get("strategy")
    
    if strategy_type == "property_scoped" and "property" in strategy:
        return _apply_property_scoped_strategy(data, strategy)
    elif strategy_type == "graph_partition":
        return create_graph_partition(data)
    elif strategy_type == "property_mapping":
        return _apply_property_mapping_strategy(data, strategy)
    elif strategy_type == "nested_contexts":
        return _apply_nested_contexts_strategy(data, strategy)
    elif strategy_type == "context_versioning":
        return _apply_context_versioning_strategy(data, strategy)
    elif strategy_type == "separate_graphs":
        return _apply_separate_graphs_strategy(data, strategy)
    
    return data


In [ ]:
#| export
def _apply_property_scoped_strategy(data, strategy):
    """Apply property-scoped context strategy"""
    import copy
    
    property_name = strategy["property"]
    if property_name in data and isinstance(data[property_name], dict):
        result = copy.deepcopy(data)
        
        # Create new context structure
        if isinstance(result["@context"], list) and len(result["@context"]) >= 2:
            new_context = {"@version": 1.1}
            
            # Use primary context as base
            primary_vocab = strategy.get("primary")
            if primary_vocab and primary_vocab in VOCABULARY_REGISTRY:
                primary_uri = VOCABULARY_REGISTRY[primary_vocab]["uri"]
                new_context["@vocab"] = primary_uri
            else:
                # Just use the first context
                new_context["@vocab"] = result["@context"][0]
            
            # Create scoped context for the property
            secondary_vocab = strategy.get("secondary")
            if secondary_vocab and secondary_vocab in VOCABULARY_REGISTRY:
                secondary_uri = VOCABULARY_REGISTRY[secondary_vocab]["uri"]
                new_context[property_name] = {
                    "@id": f"{new_context['@vocab']}{property_name}",
                    "@context": {"@vocab": secondary_uri},
                    "@protected": False
                }
            else:
                # Use second context from list
                new_context[property_name] = {
                    "@id": f"{new_context['@vocab']}{property_name}",
                    "@context": result["@context"][1],
                    "@protected": False
                }
            
            result["@context"] = new_context
            return result
    
    return data


In [ ]:
#| export
def _apply_property_mapping_strategy(data, strategy):
    """Apply property mapping strategy"""
    import copy
    
    result = copy.deepcopy(data)
    mappings = strategy.get("mappings", {})
    
    # Create a single context with mappings
    if isinstance(result["@context"], list):
        new_context = {"@version": 1.1}
        
        # Use first context as base
        if isinstance(result["@context"][0], str):
            new_context["@vocab"] = result["@context"][0]
        elif isinstance(result["@context"][0], dict):
            new_context.update(result["@context"][0])
        
        # Add mappings
        for source, target in mappings.items():
            src_prefix, src_term = source.split(":") if ":" in source else ("", source)
            new_context[src_term] = {"@id": target}
        
        result["@context"] = new_context
    
    return result


In [ ]:
#| export
def _apply_nested_contexts_strategy(data, strategy):
    """Apply nested contexts strategy"""
    import copy
    
    result = copy.deepcopy(data)
    outer = strategy.get("outer")
    inner = strategy.get("inner")
    
    if outer and inner and outer in VOCABULARY_REGISTRY and inner in VOCABULARY_REGISTRY:
        outer_uri = VOCABULARY_REGISTRY[outer]["uri"]
        inner_uri = VOCABULARY_REGISTRY[inner]["uri"]
        
        # Create nested context structure
        new_context = {
            "@version": 1.1,
            "@vocab": outer_uri,
            "inner": {
                "@id": f"{outer_uri}inner",
                "@context": {"@vocab": inner_uri}
            }
        }
        
        result["@context"] = new_context
    
    return result

In [ ]:
#| export
def _apply_context_versioning_strategy(data, strategy):
    """Apply context versioning strategy"""
    import copy
    
    result = copy.deepcopy(data)
    version = strategy.get("context_version", "1.1")
    
    # Ensure context is an object with version
    if isinstance(result["@context"], list):
        new_context = {"@version": version}
        
        # Add all contexts from the list
        for ctx in result["@context"]:
            if isinstance(ctx, str):
                # Use a generated prefix
                prefix = f"ctx{len(new_context)}"
                new_context[prefix] = ctx
            elif isinstance(ctx, dict):
                # Merge the context object
                for k, v in ctx.items():
                    if k not in new_context:
                        new_context[k] = v
        
        result["@context"] = new_context
    
    return result


In [ ]:
#| export
def _apply_separate_graphs_strategy(data, strategy):
    """Apply separate graphs strategy"""
    # This is essentially a graph partition but with specific handling
    return create_graph_partition(data)


In [ ]:
#| export
def create_graph_partition(data):
    """Create a graph partition for data with conflicting contexts"""
    if not isinstance(data, dict): return data
        
    # Start building a graph structure
    graph = {"@graph": []}
    
    # Function to process nested objects recursively
    def process_object(obj, parent_id=None, link_property=None):
        if not isinstance(obj, dict): return obj
            
        # Ensure the object has an ID
        if "@id" not in obj and "id" not in obj:
            obj_id = f"urn:uuid:{uuid.uuid4()}"
            obj["@id"] = obj_id
        else:
            obj_id = obj.get("@id", obj.get("id"))
            
        # Create a copy for the graph
        graph_obj = copy.deepcopy(obj)
        
        # Process nested objects
        for key, value in list(graph_obj.items()):
            if isinstance(value, dict) and len(value) > 1:
                # Extract nested object
                nested_id = process_object(value, obj_id, key)
                # Replace with reference
                graph_obj[key] = {"@id": nested_id}
            elif isinstance(value, list):
                # Process list of objects
                for i, item in enumerate(value):
                    if isinstance(item, dict):
                        nested_id = process_object(item)
                        value[i] = {"@id": nested_id}
        
        # Add to graph
        graph["@graph"].append(graph_obj)
        
        # If this is a child object, link back to parent
        if parent_id and link_property:
            # Find the parent in the graph
            for entity in graph["@graph"]:
                if entity.get("@id") == parent_id:
                    # Add link to this object
                    entity[link_property] = {"@id": obj_id}
                    break
        
        return obj_id
    
    # Start processing with the root object
    process_object(data)
    return graph

## Utility functions

In [ ]:
#| export
def load_context_for_vocabulary(vocab_name):
    """Load and return the context for a specific vocabulary"""
    # Check if the vocabulary is in our registry
    if vocab_name not in VOCABULARY_REGISTRY:
        raise ValueError(f"Unknown vocabulary: {vocab_name}")
    
    vocab_info = VOCABULARY_REGISTRY[vocab_name]
    vocab_uri = vocab_info["uri"]
    
    # Use our vocabulary manager to handle retrieval
    result = _manager.handle_direct_support(vocab_uri, vocab_info)
    
    # Return just the context document
    return result["document"]

In [ ]:

#| export
def create_dataset_with_vocabulary(dataset_data, vocab_name):
    """Create a dataset using a specific vocabulary"""
    # Load the context for the vocabulary
    context = load_context_for_vocabulary(vocab_name)
    
    # Create the dataset with the context
    dataset = {
        "@context": context,
        **dataset_data
    }
    
    return dataset


In [ ]:
#| export
def add_dataset_to_memory(memory, dataset_data, vocab_name):
    """Add a dataset to semantic memory with proper vocabulary handling"""
    # Register our vocabulary-aware document loader
    register_vocab_aware_loader()
    
    # Create the dataset with the vocabulary context
    dataset = create_dataset_with_vocabulary(dataset_data, vocab_name)
    
    # Add to memory
    return memory.add_jsonld(dataset)


In [ ]:
#| export
def compact_entity_with_vocabulary(entity, vocab_name):
    """Compact an entity using the proper context for a vocabulary"""
    from pyld import jsonld
    
    # Get the proper context object for this vocabulary
    context = load_context_for_vocabulary(vocab_name)
    
    # Compact using this context
    compacted = jsonld.compact(entity, context)
    
    return compacted

In [ ]:
def debug_vocabulary_detection(context_list):
    """Debug vocabulary detection for a context list"""
    print(f"\nDebugging vocabulary detection for: {context_list}")
    
    # Check each context individually
    for i, ctx in enumerate(context_list if isinstance(context_list, list) else [context_list]):
        print(f"\nContext {i+1}: {ctx}")
        
        # See if we can match it to known vocabularies
        for vocab_name, vocab_info in VOCABULARY_REGISTRY.items():
            vocab_uri = vocab_info.get("uri", "")
            if isinstance(ctx, str):
                if ctx == vocab_uri:
                    print(f"  EXACT MATCH with {vocab_name}: {vocab_uri}")
                elif ctx.startswith(vocab_uri):
                    print(f"  PREFIX MATCH with {vocab_name}: {vocab_uri}")
        
        # Try to extract a URI from the context
        if isinstance(ctx, str):
            print(f"  URI: {ctx}")
        elif isinstance(ctx, dict) and "@vocab" in ctx:
            print(f"  @vocab: {ctx['@vocab']}")
    
    # Run the actual detection
    vocabs = detect_vocabularies_in_context(context_list)
    print(f"\nDetected vocabularies: {vocabs}")
    
    # Check collision strategies
    if len(vocabs) >= 2:
        vocabs.sort()
        vocab_pair = tuple(vocabs[:2])
        print(f"Checking for collision strategy for: {vocab_pair}")
        
        if vocab_pair in COLLISION_STRATEGIES:
            print(f"  Strategy found: {COLLISION_STRATEGIES[vocab_pair]['strategy']}")
        else:
            print("  No direct strategy found")
            
            # Check reversed
            reversed_pair = (vocab_pair[1], vocab_pair[0])
            if reversed_pair in COLLISION_STRATEGIES:
                print(f"  Strategy found for reversed pair: {COLLISION_STRATEGIES[reversed_pair]['strategy']}")
    
    return vocabs


In [ ]:
# Define the test contexts
test_contexts = ["https://www.w3.org/ns/credentials/v2", "https://gs1.org/epcis/"]

# Debug the vocabulary detection
debug_vocabulary_detection(test_contexts)



Debugging vocabulary detection for: ['https://www.w3.org/ns/credentials/v2', 'https://gs1.org/epcis/']

Context 1: https://www.w3.org/ns/credentials/v2
  EXACT MATCH with vc: https://www.w3.org/ns/credentials/v2
  URI: https://www.w3.org/ns/credentials/v2

Context 2: https://gs1.org/epcis/
  URI: https://gs1.org/epcis/

Detected vocabularies: ['epcis', 'vc']
Checking for collision strategy for: ('epcis', 'vc')
  No direct strategy found
  Strategy found for reversed pair: property_scoped


['epcis', 'vc']

## Tests

In [ ]:
def test_vocabulary_tools_refactoring():
    """Test the refactored vocabulary tools functionality"""
    print("=== Testing Vocabulary Tools Refactoring ===\n")
    
    # 1. Test vocabulary registry structure
    print("1. Testing vocabulary registry structure...")
    for vocab_name, vocab_info in VOCABULARY_REGISTRY.items():
        assert "uri" in vocab_info, f"Missing URI for {vocab_name}"
        assert "prefix" in vocab_info, f"Missing prefix for {vocab_name}"
        assert "resources" in vocab_info, f"Missing resources for {vocab_name}"
        assert "features" in vocab_info, f"Missing features for {vocab_name}"
    print("✓ Vocabulary registry structure valid\n")
    
    # 2. Test vocabulary manager initialization
    print("2. Testing vocabulary manager initialization...")
    manager = VocabularyManager()
    assert manager.registry == VOCABULARY_REGISTRY, "Registry not properly initialized"
    assert isinstance(manager.context_cache, dict), "Context cache not initialized"
    print("✓ Vocabulary manager initializes correctly\n")
    
    # 3. Test URL transformations
    print("3. Testing URL transformations...")
    # Test a URL that should be transformed (EPCIS)
    gs1_url = "https://gs1.github.io/EPCIS/cbv-context-disposition.jsonld"
    epcis_info = VOCABULARY_REGISTRY["epcis"]
    transformed = manager.apply_url_transformations(gs1_url, epcis_info)
    assert transformed != gs1_url, "URL transformation failed"
    assert "raw.githubusercontent.com" in transformed, "Incorrect transformation"
    print(f"Original: {gs1_url}")
    print(f"Transformed: {transformed}")
    print("✓ URL transformations working\n")
    
    # 4. Test context loading
    print("4. Testing context loading...")
    # Try loading a well-known vocabulary
    try:
        schema_context = load_context_for_vocabulary("schema")
        assert schema_context is not None, "Failed to load schema.org context"
        assert "@context" in schema_context or "@vocab" in schema_context, "Invalid context structure"
        print("✓ Successfully loaded schema.org context")
    except Exception as e:
        print(f"✗ Failed to load schema.org context: {e}")
    print("")
    
    # 5. Test collision strategy detection
    print("5. Testing collision strategy detection...")
    # Test a known collision pair
    test_contexts = ["https://www.w3.org/ns/credentials/v2", "https://gs1.org/epcis/"]
    strategy = get_collision_strategy(test_contexts)
    assert strategy is not None, "Failed to detect collision strategy"
    assert strategy["strategy"] == "property_scoped", "Wrong collision strategy detected"
    assert strategy["primary"] == "vc", "Wrong primary vocabulary"
    assert strategy["secondary"] == "epcis", "Wrong secondary vocabulary"
    print(f"Detected strategy: {strategy['strategy']} (primary: {strategy['primary']}, secondary: {strategy['secondary']})")
    print("✓ Collision strategy detection working\n")
    
    # 6. Test applying collision strategy
    print("6. Testing applying collision strategy...")
    # Create test data with VC and EPCIS
    test_data = {
        "@context": [
            "https://www.w3.org/ns/credentials/v2",
            "https://gs1.org/epcis/"
        ],
        "@id": "urn:uuid:test-credential",
        "@type": ["VerifiableCredential"],
        "issuer": "https://example.org/issuer",
        "credentialSubject": {
            "@id": "urn:epc:id:sgtin:0614141.107346.2017",
            "@type": "ObjectEvent",
            "eventTime": "2019-04-02T15:00:00.000Z"
        }
    }
    
    # Apply the strategy
    result = apply_collision_strategy(test_data, strategy)
    assert "@context" in result, "Context missing after strategy application"
    assert isinstance(result["@context"], dict), "Context should be an object after property_scoped strategy"
    assert "@version" in result["@context"], "Context should have @version after strategy"
    assert "credentialSubject" in result["@context"], "Context should have credentialSubject definition"
    print("✓ Applied property_scoped strategy successfully\n")
    
    # 7. Test graph partitioning
    print("7. Testing graph partitioning...")
    # Create test data with nested objects
    test_graph_data = {
        "@context": [
            "https://schema.org/",
            "https://www.w3.org/ns/activitystreams"
        ],
        "@id": "urn:test:organization",
        "@type": "Organization",
        "name": "Test Organization",
        "department": {
            "@id": "urn:test:department",
            "@type": "Department",
            "name": "Test Department",
            "employee": {
                "@id": "urn:test:employee",
                "@type": "Person",
                "name": "Test Employee"
            }
        }
    }
    
    # Create graph partition
    partitioned = create_graph_partition(test_graph_data)
    assert "@graph" in partitioned, "Graph structure missing"
    assert len(partitioned["@graph"]) == 3, f"Expected 3 entities, got {len(partitioned['@graph'])}"
    
    # Check that entities are properly linked
    org = next((e for e in partitioned["@graph"] if e["@id"] == "urn:test:organization"), None)
    dept = next((e for e in partitioned["@graph"] if e["@id"] == "urn:test:department"), None)
    emp = next((e for e in partitioned["@graph"] if e["@id"] == "urn:test:employee"), None)
    
    assert org is not None, "Organization entity missing"
    assert dept is not None, "Department entity missing"
    assert emp is not None, "Employee entity missing"
    
    assert "department" in org, "Department reference missing from organization"
    assert org["department"]["@id"] == "urn:test:department", "Incorrect department reference"
    
    assert "employee" in dept, "Employee reference missing from department"
    assert dept["employee"]["@id"] == "urn:test:employee", "Incorrect employee reference"
    
    print("✓ Graph partitioning working correctly\n")
    
    # 8. Test document loader creation
    print("8. Testing document loader creation...")
    loader = create_vocab_aware_document_loader()
    assert callable(loader), "Document loader is not callable"
    print("✓ Document loader created successfully\n")
    
    # 9. Test vocabulary detection
    print("9. Testing vocabulary detection...")
    # Test with string context
    schema_vocabs = detect_vocabularies_in_context("https://schema.org/")
    assert "schema" in schema_vocabs, "Failed to detect schema.org from string"
    
    # Test with object context
    object_context = {
        "@vocab": "https://schema.org/",
        "vc": "https://www.w3.org/ns/credentials/v2"
    }
    object_vocabs = detect_vocabularies_in_context(object_context)
    assert "schema" in object_vocabs, "Failed to detect schema.org from @vocab"
    
    # Test with list context
    list_context = [
        "https://schema.org/",
        "https://www.w3.org/ns/credentials/v2"
    ]
    list_vocabs = detect_vocabularies_in_context(list_context)
    assert "schema" in list_vocabs, "Failed to detect schema.org from list"
    assert "vc" in list_vocabs, "Failed to detect vc from list"
    
    print(f"Detected vocabularies from list context: {', '.join(list_vocabs)}")
    print("✓ Vocabulary detection working correctly\n")
    
    # 10. Test dataset creation
    print("10. Testing dataset creation with vocabulary...")
    dataset_data = {
        "@id": "https://example.org/datasets/test",
        "@type": "Dataset",
        "name": "Test Dataset",
        "description": "A dataset for testing"
    }
    
    dataset = create_dataset_with_vocabulary(dataset_data, "schema")
    assert "@context" in dataset, "Context missing in created dataset"
    assert dataset["@id"] == "https://example.org/datasets/test", "Dataset ID incorrect"
    assert dataset["@type"] == "Dataset", "Dataset type incorrect"
    print("✓ Dataset creation with vocabulary working\n")
    
    # Final summary
    print("=== All Vocabulary Tools Tests Passed ===")
    
    return {
        "manager": manager,
        "transformed_url": transformed,
        "strategy": strategy,
        "applied_strategy": result,
        "partitioned_graph": partitioned,
        "dataset": dataset
    }


In [ ]:
man, turi, strat, appstrat, pg, ds = test_vocabulary_tools_refactoring()

=== Testing Vocabulary Tools Refactoring ===

1. Testing vocabulary registry structure...
✓ Vocabulary registry structure valid

2. Testing vocabulary manager initialization...
✓ Vocabulary manager initializes correctly

3. Testing URL transformations...
--- [Transform] Applied transformation: https://gs1.github.io/EPCIS/cbv-context-disposition.jsonld -> https://raw.githubusercontent.com/gs1/EPCIS/master/JSON-LD-Context/cbv-context-disposition.jsonld
Original: https://gs1.github.io/EPCIS/cbv-context-disposition.jsonld
Transformed: https://raw.githubusercontent.com/gs1/EPCIS/master/JSON-LD-Context/cbv-context-disposition.jsonld
✓ URL transformations working

4. Testing context loading...
--- [Direct] handle_direct_support for: https://schema.org/
--- [Direct] Cache MISS for https://schema.org/
--- [Direct] Context location: https://schema.org/docs/jsonldcontext.jsonld
--- [Direct] Fetching https://schema.org/docs/jsonldcontext.jsonld...
--- [Direct] Status code for https://schema.org/do

## Examples and tests using SM (not exported to avoid circular imports)

In [ ]:
def example_croissant_usage(memory):
    """Example of using CROISSANT for ML dataset metadata"""
    # Create a comprehensive CROISSANT dataset
    dataset_data = {
        "@type": "Dataset",
        "@id": "https://example.org/datasets/mnist",
        "name": "MNIST Handwritten Digits",
        "description": "Database of handwritten digits with 60,000 training examples and 10,000 test examples",
        "url": "https://example.org/datasets/mnist",
        "version": "1.0.0",
        "license": "https://creativecommons.org/licenses/by/4.0/",
        "datePublished": "2021-01-15",
        "creator": {
            "@type": "Person",
            "name": "Yann LeCun",
            "affiliation": {
                "@type": "Organization",
                "name": "New York University"
            }
        },
        "distribution": {
            "@type": "DatasetDistribution",
            "contentUrl": "https://example.org/downloads/mnist.zip",
            "encodingFormat": "application/zip",
            "sha256": "a02ffd6da8d7f56925c13a796e5609ac22b3bc61"
        },
        "keywords": ["computer vision", "image recognition", "handwritten digits", "machine learning"],
        "citation": "LeCun, Y., Cortes, C., & Burges, C. (1998). The MNIST database of handwritten digits.",
        "isAccessibleForFree": True,
        "recordedAt": {
            "@type": "Event",
            "name": "MNIST Data Collection",
            "startDate": "1998-01-01"
        }
    }
    
    # Add to memory using the CROISSANT vocabulary
    expanded = add_dataset_to_memory(memory, dataset_data, "croissant")
    
    # Return the dataset ID for reference
    return dataset_data["@id"]

def example_vc_usage(memory):
    """Example of using Verifiable Credentials vocabulary"""
    # Create a Verifiable Credential
    vc_data = {
        "@type": "VerifiableCredential",
        "@id": "https://example.org/credentials/3732",
        "issuer": {
            "@id": "https://example.org/issuers/14",
            "@type": "Organization",
            "name": "Example University",
            "url": "https://example.org"
        },
        "issuanceDate": "2023-06-15T16:20:10Z",
        "expirationDate": "2028-06-14T23:59:59Z",
        "credentialSubject": {
            "@id": "https://example.org/people/alice",
            "@type": "Person",
            "name": "Alice Johnson",
            "degree": {
                "@type": "BachelorDegree",
                "name": "Bachelor of Science in Computer Science",
                "awardedDate": "2023-05-30"
            }
        },
        "proof": {
            "@type": "Ed25519Signature2020",
            "created": "2023-06-15T16:20:10Z",
            "verificationMethod": "https://example.org/issuers/14#key-1",
            "proofPurpose": "assertionMethod",
            "proofValue": "z58DAdFfa9SkqZMVPxAQpic7ndSayn1PzZs6ZjWp1CktyGesjuTdwCBxNJ2dQQ92coPcnMqLHVdJuiCwHGGAXnJF"
        }
    }
    
    # Add to memory using the VC vocabulary
    expanded = add_dataset_to_memory(memory, vc_data, "vc")
    
    # Return the credential ID
    return vc_data["@id"]

def example_schema_usage(memory):
    """Example of using Schema.org vocabulary for rich content"""
    # Create a complex Schema.org entity
    recipe_data = {
        "@type": "Recipe",
        "@id": "https://example.org/recipes/chocolate-cake",
        "name": "Chocolate Cake",
        "image": "https://example.org/images/chocolate-cake.jpg",
        "description": "A rich and moist chocolate cake that's perfect for any occasion.",
        "prepTime": "PT20M",
        "cookTime": "PT35M",
        "totalTime": "PT55M",
        "recipeYield": "12 servings",
        "recipeCategory": "Dessert",
        "recipeCuisine": "American",
        "nutrition": {
            "@type": "NutritionInformation",
            "calories": "350 calories",
            "fatContent": "18 g",
            "saturatedFatContent": "9 g",
            "carbohydrateContent": "45 g",
            "sugarContent": "30 g",
            "proteinContent": "5 g"
        },
        "recipeIngredient": [
            "2 cups all-purpose flour",
            "2 cups sugar",
            "3/4 cup unsweetened cocoa powder",
            "2 teaspoons baking soda",
            "1 teaspoon salt",
            "2 eggs",
            "1 cup buttermilk",
            "1/2 cup vegetable oil",
            "2 teaspoons vanilla extract",
            "1 cup hot coffee"
        ],
        "recipeInstructions": [
            {
                "@type": "HowToStep",
                "text": "Preheat oven to 350°F (175°C). Grease and flour two 9-inch round cake pans."
            },
            {
                "@type": "HowToStep",
                "text": "In a large bowl, combine flour, sugar, cocoa, baking soda, and salt."
            },
            {
                "@type": "HowToStep",
                "text": "Add eggs, buttermilk, oil, and vanilla; beat for 2 minutes."
            },
            {
                "@type": "HowToStep",
                "text": "Stir in hot coffee. The batter will be thin."
            },
            {
                "@type": "HowToStep",
                "text": "Pour into prepared pans and bake for 30-35 minutes."
            },
            {
                "@type": "HowToStep",
                "text": "Cool and frost with your favorite chocolate frosting."
            }
        ],
        "author": {
            "@type": "Person",
            "name": "Chef Julia",
            "url": "https://example.org/chefs/julia"
        },
        "aggregateRating": {
            "@type": "AggregateRating",
            "ratingValue": "4.8",
            "reviewCount": "234"
        },
        "keywords": "chocolate, cake, dessert, baking",
        "suitableForDiet": "VegetarianDiet"
    }
    
    # Add to memory using the Schema.org vocabulary
    expanded = add_dataset_to_memory(memory, recipe_data, "schema")
    
    # Return the recipe ID
    return recipe_data["@id"]

def example_epcis_usage(memory):
    """Example of using EPCIS vocabulary for supply chain events"""
    # Create an EPCIS event
    epcis_data = {
        "@type": "ObjectEvent",
        "@id": "https://example.org/epcis/events/1234567890",
        "eventTime": "2023-10-15T13:45:30.000Z",
        "eventTimeZoneOffset": "+02:00",
        "epcList": ["urn:epc:id:sgtin:0614141.107346.2017", "urn:epc:id:sgtin:0614141.107346.2018"],
        "action": "OBSERVE",
        "bizStep": "shipping",
        "disposition": "in_transit",
        "readPoint": {
            "@id": "urn:epc:id:sgln:0614141.00777.0",
            "site": "Distribution Center 1"
        },
        "bizLocation": {
            "@id": "urn:epc:id:sgln:0614141.00888.0",
            "address": {
                "@type": "PostalAddress",
                "streetAddress": "123 Supply Chain Rd",
                "addressLocality": "Logistics City",
                "addressRegion": "LC",
                "postalCode": "12345",
                "addressCountry": "US"
            }
        },
        "bizTransactionList": [
            {
                "type": "po",
                "@id": "urn:epcglobal:cbv:bt:0614141000001:1234567890"
            },
            {
                "type": "desadv",
                "@id": "urn:epcglobal:cbv:bt:0614141000001:DE9876"
            }
        ],
        "sourceList": [
            {
                "type": "owning_party",
                "source": "urn:epc:id:sgln:0614141.00001.0"
            }
        ],
        "destinationList": [
            {
                "type": "location",
                "destination": "urn:epc:id:sgln:0614141.00777.0"
            }
        ],
        "sensorElementList": [
            {
                "sensorMetadata": {
                    "time": "2023-10-15T13:45:30.000Z",
                    "deviceID": "urn:epc:id:giai:4000001.111",
                    "deviceMetadata": "https://example.org/devices/111",
                    "rawData": "https://example.org/devices/111/data"
                },
                "sensorReport": [
                    {
                        "type": "Temperature",
                        "value": 26.0,
                        "uom": "CEL",
                        "deviceID": "urn:epc:id:giai:4000001.111"
                    },
                    {
                        "type": "Humidity",
                        "value": 12.1,
                        "uom": "A93",
                        "deviceID": "urn:epc:id:giai:4000001.111"
                    }
                ]
            }
        ]
    }
    
    # Add to memory using the EPCIS vocabulary
    expanded = add_dataset_to_memory(memory, epcis_data, "epcis")
    
    # Return the event ID
    return epcis_data["@id"]

def example_rocrate_usage(memory):
    """Example of using RO-Crate vocabulary for research objects"""
    # Create an RO-Crate for a research dataset
    rocrate_data = {
        "@type": "Dataset",
        "@id": "./",
        "name": "Climate Change Research Dataset",
        "description": "A comprehensive dataset of climate measurements from 1950-2020",
        "license": "https://creativecommons.org/licenses/by/4.0/",
        "datePublished": "2023-01-30",
        "hasPart": [
            {
                "@id": "data/temperature.csv",
                "@type": "File",
                "name": "Global Temperature Measurements",
                "contentSize": "2.4 MB",
                "encodingFormat": "text/csv"
            },
            {
                "@id": "data/precipitation.csv",
                "@type": "File",
                "name": "Global Precipitation Measurements",
                "contentSize": "3.1 MB",
                "encodingFormat": "text/csv"
            },
            {
                "@id": "code/analysis.py",
                "@type": "SoftwareSourceCode",
                "name": "Climate Data Analysis Script",
                "programmingLanguage": {
                    "@id": "https://www.python.org/",
                    "@type": "ComputerLanguage",
                    "name": "Python",
                    "version": "3.9"
                }
            }
        ],
        "author": [
            {
                "@id": "https://orcid.org/0000-0002-1825-0097",
                "@type": "Person",
                "name": "Dr. Jane Smith",
                "affiliation": {
                    "@id": "https://ror.org/03f0f6041",
                    "@type": "Organization",
                    "name": "Climate Research Institute"
                }
            },
            {
                "@id": "https://orcid.org/0000-0001-5225-4203",
                "@type": "Person",
                "name": "Dr. John Brown",
                "affiliation": {
                    "@id": "https://ror.org/03f0f6041",
                    "@type": "Organization",
                    "name": "Climate Research Institute"
                }
            }
        ],
        "citation": "Smith, J., & Brown, J. (2023). Climate Change Research Dataset. Climate Research Institute. https://doi.org/10.5281/zenodo.1234567",
        "funder": {
            "@id": "https://ror.org/00d4ym594",
            "@type": "Organization",
            "name": "National Science Foundation"
        }
    }
    
    # Add to memory using the RO-Crate vocabulary
    expanded = add_dataset_to_memory(memory, rocrate_data, "ro-crate")
    
    # Return the dataset ID
    return rocrate_data["@id"]


In [ ]:
def test_examples():
    """Test the examples with multiple vocabularies"""
    from cogitarelink.memory import SemanticMemory
    
    print("=== Testing Examples with Multiple Vocabularies ===")
    
    # Initialize memory
    memory = SemanticMemory()
    
    # Register our vocabulary-aware document loader
    register_vocab_aware_loader()
    
    # 1. Add datasets with different vocabularies
    print("\n1. Adding datasets with different vocabularies:")
    
    print("Adding CROISSANT dataset...")
    croissant_id = example_croissant_usage(memory)
    
    print("Adding Schema.org dataset...")
    schema_id = example_schema_usage(memory)
    
    print("Adding Verifiable Credential...")
    vc_id = example_vc_usage(memory)
    
    print("Adding EPCIS event...")
    epcis_id = example_epcis_usage(memory)
    
    print("Adding RO-Crate dataset...")
    rocrate_id = example_rocrate_usage(memory)
    
    # 2. Retrieve and compact entities
    print("\n2. Retrieving and compacting entities:")
    
    # Get entities from memory
    croissant_entity = memory.query_by_id(croissant_id)
    schema_entity = memory.query_by_id(schema_id)
    vc_entity = memory.query_by_id(vc_id)
    epcis_entity = memory.query_by_id(epcis_id)
    rocrate_entity = memory.query_by_id(rocrate_id)
    
    # Compact with appropriate vocabularies
    print("Compacting CROISSANT dataset...")
    compacted_croissant = compact_entity_with_vocabulary(croissant_entity, "croissant")
    
    print("Compacting Schema.org dataset...")
    compacted_schema = compact_entity_with_vocabulary(schema_entity, "schema")
    
    print("Compacting Verifiable Credential...")
    compacted_vc = compact_entity_with_vocabulary(vc_entity, "vc")
    
    print("Compacting EPCIS event...")
    compacted_epcis = compact_entity_with_vocabulary(epcis_entity, "epcis")
    
    print("Compacting RO-Crate dataset...")
    compacted_rocrate = compact_entity_with_vocabulary(rocrate_entity, "ro-crate")
    
    # 3. Verify properties are accessible in compacted form
    print("\n3. Verifying properties in compacted form:")
    
    print(f"CROISSANT dataset name: {compacted_croissant.get('name', 'Unknown')}")
    print(f"Schema.org recipe name: {compacted_schema.get('name', 'Unknown')}")
    print(f"VC issuer: {compacted_vc.get('issuer', {}).get('name', 'Unknown')}")
    print(f"EPCIS event time: {compacted_epcis.get('eventTime', 'Unknown')}")
    print(f"RO-Crate description: {compacted_rocrate.get('description', 'Unknown')}")
    
    # 4. Test cross-vocabulary search
    print("\n4. Testing cross-vocabulary search:")
    
    name_results = memory.search("Chocolate")
    print(f"Search for 'Chocolate' found {len(name_results)} results")
    
    org_results = memory.search("Organization")
    print(f"Search for 'Organization' found {len(org_results)} results")
    
    # 5. Test loaded vocabularies
    print("\n5. Checking loaded vocabularies:")
    loaded_vocabs = get_loaded_vocabularies()
    print(f"Loaded vocabularies: {', '.join(loaded_vocabs)}")
    
    print("\n=== Examples Test Complete ===")
    
    return {
        "croissant": compacted_croissant,
        "schema": compacted_schema,
        "vc": compacted_vc,
        "epcis": compacted_epcis,
        "rocrate": compacted_rocrate
    }


In [ ]:
cr, sc, v, ep, ro = test_examples()

=== Testing Examples with Multiple Vocabularies ===

1. Adding datasets with different vocabularies:
Adding CROISSANT dataset...
--- [Direct] handle_direct_support for: http://mlcommons.org/croissant/
--- [Direct] Cache MISS for http://mlcommons.org/croissant/
--- [Direct] Context location: https://raw.githubusercontent.com/mlcommons/croissant/main/docs/context.jsonld
--- [Direct] Fetching https://raw.githubusercontent.com/mlcommons/croissant/main/docs/context.jsonld...
--- [Direct] Status code for https://raw.githubusercontent.com/mlcommons/croissant/main/docs/context.jsonld: 404
--- [Direct] HTTP Status Error fetching context from https://raw.githubusercontent.com/mlcommons/croissant/main/docs/context.jsonld: 404 - 404: Not Found...
--- [Direct] Trying fallbacks: []
--- [Direct] All fetch attempts failed for http://mlcommons.org/croissant/. Creating minimal.
--- [Minimal] Creating minimal context for http://mlcommons.org/croissant/


AttributeError: 'SemanticMemory' object has no attribute 'add_jsonld'

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()